# 📋 MNT02 미션 09 — AAR 기반 부품 과다/과소 재고 예측 오차 분석 및 안전재고 파라미터 재보정

**Solution 교관 전용** / 학생 공개 첨부 금지 / 교육용 가상 자료

---

## 시작 전 꼭 읽으세요

이번 활동은 **교육용 가상 부품 데이터**를 분석하는 실습입니다. 코딩이 처음이어도 안내와 TODO를 따라 한 단계씩 진행하세요.

1. 아래 **준비 셀**을 실행합니다. 필요한 라이브러리는 실습 환경에 설치되어 있어야 합니다.
2. 미션의 입력 자료·핵심 함수를 확인합니다. 학생용 `TODO` 부분을 직접 작성하거나 허용된 AI에 도움을 요청합니다.
3. 코드를 실행하고 실제 출력·오류를 기록합니다. 실행이 실패했다면 실패 내용을 그대로 남기고 수정 후 다시 실행하세요.
4. AI에게 보낸 **실제 질문과 받은 답변 원문**을 작성 기록에 붙여넣고, 코드·출력과 비교한 수정 이유를 작성합니다.
5. 마지막 기록 셀을 실행해서 JSON과 검토용 ZIP을 만들고 내려받습니다. 파일 생성만으로 학습 PASS나 사람의 승인이 되지 않습니다.

**인터넷/폐쇄망:** Colab과 외부 AI는 접속이 허용된 환경에서만 사용합니다.

폐쇄망에서는 사전 설치된 로컬 Jupyter 등 허용된 환경이 필요하며, 외부 AI 사용이 불가능하면 '미실시'로 기록하세요.

**보안:** 실제 부대·조달·업체·정비 자료를 입력하거나 AI로 실제 발주를 승인하지 마세요.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'MNT02_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
MNT02_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global MNT02_PROJECT_READY, MNT02_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        members = [m for m in archive.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('ZIP 안에 실습자료 파일이 없습니다.')
        if len(members) > 15000 or sum(m.file_size for m in members) > 800*1024*1024:
            raise ValueError('ZIP이 너무 크거나 파일 개수가 너무 많습니다.')
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    MNT02_PROJECT_DIR = WORKSPACE
    MNT02_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 4개: MNT02_Colab_실행안내.md, MNT02_M09_forms.md, MNT02_M09_lesson.md, MNT02_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVz+D1bE0QYAANcMAAAbAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVdNTxtXFN3zK55UdVkH8lW6RCmqLAFFcbLpBpngSG7B'
    'tmyzSFcGBuRgU0ywwziMydCYzzrN4A88qGbjnzPvzX/oue8920PCAhkzb+4799xzz718x2bnXow/ZM+Sy9FFJgp1//1bJip5vt4eG5sIqacLmezqUiyRDcVT'
    'bxKL/KTHxEHba9lM1HP8xmBiMycOSsKw9HGEEdsdv2zwY0vYNX/bDf0ZT4kaTpoNfnrLC3leqIeYsA3RbAvbEkd7vOWy38LzXqvHvE6DXxt+5WJwznMdcdxm'
    'Im8Ku8L83TbfKfPtMhNHDYXYr7T5rsH4zr/ifVvsO+IshyS2AUKHGHsYYn7FEB/KfsVivNYTtTZ3i/wj4DT4WZEJoy5uzMCdidWV1Jt+NxVNLEUz/W7mVZy+'
    '4uOPePaH5Vg0neh3V6LZ1HIyuxxflLmtN8SHi0AIUXX5dY5IRYoA5+BS33D63an5MPPX6/2uODqR/B04XpOeDjGa32bwKMT4F1dsmmwmlskkcf3rZHolI2om'
    'EeabBvNahqjcSii1WyoPOBU1l04QNttgfrHEC+cAEEa9GD879lwXh3vyHbsSxK5AT4UZ8Ijm536XFzq8lQPm5j9MtPPQSL/rdXvelcN4MSeOkRdvtUXVwJGO'
    'Jf7eBKe2v1Gj2PTguD0KP/Z4kAxvGVoGbCn2mqQVX0ktx1agtmg2nkww3nDFUQl6I8TQiTCuUG2WjqWWo69iC69XE6/onLB6EBtC5cBpMA/U+ZQg1SgMELsO'
    'WPGcd6IGgi0bPN8jtNsy37dIYaHFGFiOUWiVAy+ZUvhHJr/8j5cswsebFVxgWPxyk6KCEZSdl4ZR55LZ8CCp2NJ0Op1MM7Gx5m9YqmuklKWOSQKivocajV4f'
    'e0LS7YhCAx/n6IFBuf1Dg6VXEwsr8UyGGKAaNk1RK7JhUzDICh1FtH5dAmKGIgxSvi57Tg6ncl7zFkrqofqSUEkF0gNPe/jEU8gsGPfG8DqlQFzUVQMkGMVz'
    'aRfGFd8+kXfVc0S+rtY9naq7XLapUsh1Dk0T7PUhM09DLLmaTa1mMw9meTPvm+4D/0NROOeklUx2CQ/RstmlWDpNyAmySqLr8tNevwv2Qr9niDoQAKUC5ahq'
    'fHcTVsX45Tnfz8u6OjlUDUBK/FORsvKcNXzRb6Jn+ZeczKpypKHLzpME6qzkRXh/t6oZkNHtwXFSuL0GE4HKQIzUxR1uvFbH3+roKsmCHzTgYQOQw/cJzEV7'
    'UE3eLAsLPcCbBrVaTRadH9cCTP4Ip/+RibM1Ye8B4Sa3TwKuPsjcrqDa6C8UHDd/9is2cnjkXbVJTHTCa+b8LTsoMviOZUM+Zl6swyFQIr9qgi6mrINo9Kvg'
    'xQVkaqyZ6annc9PPF8Jz8y9f4OISImi49xkJo8AFi0XCsy9npl5M/0zKhFSk6FQ2yvOG/mCJa+s+fx04nnJxWVdYxmkNtDmW+ETh9mSH4r2DrYAvT4K6SVT6'
    '0fj491Qxf/ctENEfuOMQlXeGnbTZagXTSlQhKdVNhKrqjh5JcekhKFmBPnihQXer3yiVr2vIqFEqEJx7LJVGzdyhjPx3eX55IecEBgRVAanQr7DGC2eY9+EN'
    'KlEm2XjOlSjYw0DAXDF4oTpI+CckPDFO8Uh2O672eSo++JIXq9ZRYwB8QD9kLnfCYK60hbkmtQUg+9bEuOcAxtGJ7Bo9M1su/BWAMSDJdbqW17yQcwaskHNS'
    'mfVMGEni2oJ80UJUDcrcOVfLAp3DUDbQvt/CoTlDUu80MMpkLutSReg3sg29YKy3IQ5qMtyHaIplgx4p6Y/qPTYxLueJnMG5iXE9uwcqyJskpoA6Dku0AWkq'
    'Zx/3u7NP8TOpJjPJQ/d0ZHpm+hmEvjAbjkTCv85FSEcUXEbUDScFpPUhBQSnF1s72rugbBNdJPEbw/Yb8vdsNiL3kLz5QMPRpaIdaaAsmbN6NZgzNkZ+lifv'
    'OnsrDQSH9fBSQ4EGJcWUo1NTopZOhZfEKdtW8UnmAr+WNgDRw7gpXb9YxNLCKAfV6ncsZBU7WXQJI4F20cHiiR2P738OzL5BidXs1bsOFhn0QEBgENMFHiNZ'
    'eFelSlutlDObn4pEAKxT8tpQ0vaN3LMM7Rn3+4uepMPYtIbJujfkq8F1We+4cknTjge63tVJgNyuc+cwYEBjSjtgbiPPfolnmf+Xq7X5cDL0BGbkOUZAaqiv'
    'bLGtIi0T6rFenOQ8uZaDh+9foRxyyfhGIQPLQAQshyr+SBXKN4KCkv9b8I+Ov4FLDNoeUWwQLbtBDi21tanVSgXX/5CgxzFytrErS1lqgxo5TdmRtn5m3N0M'
    '/gdQSwMEFAAAAAgAAAAhXEfLYvCSBwAArxEAABIAAABNTlQwMl9NMDlfZm9ybXMubWTVWEFPG0cUvvMrRsqllQhOALWqLxVtL5GaHqreqgqhlFRIJUQYRUqE'
    'IgNr5GA3GOolC1m76wQCpE5ZwOBFMj3wU5Lbzux/6PfezKztpKGXplKVKLZ3d2be+973vvdtroib33x3bVjcvPaZUPU15RwKtbGmSjXxOl8VY2PfijgKZegJ'
    'eZpP1osiPu7I0nYGH2q5jBXN+DgQyiuqqIGPbRXu4UlHOTUhw1Wh3KIKHPNUUi7LWkceRIkT0kp53FKBOzCgnlbkcSSS1UdxmM+K7xPXUUXvBz48feyinWzt'
    '4VS7O4eXPI7oQGygapG5M9hd03eiqnkiPsqrFzV8hNhpUIzIcl7gOfnbI2F2V7v5ZM1Xm3khDzpxWJGrzqCQJdxz5LEj8FPVC0I2fPmrL3dPVM3BBr4q+clG'
    'Ua5UkcDQwIA8zsdH5wQV7mUBQl18Eoc+4qzK3bLyO1gvxr74Ejk9jpKNnYv2A7nToUj0LgiCQ11sJq5HyCFgwl4tV1RpW+AQSk6fQmnJ9e30AJMc7SfPnPik'
    'krg+CoaorlwhRC/aPcjPA6uKfF6OT5px6Ih5HdSDbly5qZ+mJ3C99wQNES5SHEsLBE6YT9Gih3VggZM8dbCV6wM8MY/Drl69Kt77L+6Lvr8f5gLDcINInWz9'
    'ctFGTjL08WmYAa43agMDVwXARwZgUAUJpp1R8lVQyYrXGw3aRJZOkCvybWGdoMfphrmqnrRkvSyAgVmNTcduqCcVIXcbcRThgYJstohqqtZSfpClUwOAdZK4'
    'gSkMqnVeBdXwCZa550zRp6ipB5qYUtM69A86APtwKy4tAHRzl/dHQ+0uCPlyT+60aCEz61WWCIAWSDaruLevggVABILhKKG2quroFZ0lm+gsr8LLgryq7wjl'
    'O+rM6z5rE770WRsh2LpfRSSXEWLgkhpCLji0i7Y+VtXyacwcALUk8bvRStx9WSoS+YVG0TDThtinBtwwoac8xo422HQMCsRqTYzujgQdGKCcbfmykOasT1FO'
    'kCzVxHy3hqXtZOMRWqFQVivbyUok4nYnPgz7YNFM4wIzHWi9H6DZsL7osQIiKrnDwJ/46lnBxEsqp2odPGaw/hCdNjASH7Y4sqUFRricV42wF2HOVcijYuJF'
    'rGbhnxrkJeJ83I4QGwmb3K3wMHBbsnQm5GJLbe1jX7no2TpCLtRiaGHrqQ8YqlZOeiqg68lkR31WIFh9M8Y+wK1pG0keOfQzvccN+vY9LV6iL4Ie3mv4TZ1t'
    'uf4F2EmczDREQmo7z8ea0yAApz5lrQclEHyFriUd1o+G60jg1EFvQc6iUAu6hZQzwPWUd6UmJAkLDiJ8JdwqPFNOi7T1vG2A/0a1SfiQZKuYFsL8pA5rFWEi'
    'TEebErBS2pIVIjSUPF3g9gCzlENy3o7kC+SLQYuvBrue1SCQqoGtB5EqeGYgS8cjWdbTlSaCFVcM0bjFEzlZhjBDXZd8PQPQBaT8/LVXMfiKWjlDa7/JV2WJ'
    'qI4GQV61DmaIrcObvMslT9wTVWrinGbiOekMQqHdPbQ+C0lBBmA3ksCnSDZd8ju4fvS7sOKiRUL00LarvaEfHzWpTeDQoCDpw5YOl0rx5D384D9wNgeRiZLO'
    'KO+x6jmHaL0uC3kKuY4sbabaQGE6h5hsJjSm9Sm5pb4u5HusLsa5dNWWUNr05eomuTC163DKbw3tefOImLcI6Zh62oDz7SqlKSZT5x/ojjvX8Z1sMvQJ3iGr'
    'zXELbKlA4UA7440RV6855gEMuLeq/TaZBw47VmPy3uuVWfqsaYNuWheqLSvfXQxBDeP0SG7JQm9VueKGOloRaGruIXtAAH+ztU9sp+JpY0NlmbyHYYRFhmqk'
    'xe4521wedSbWdGxpm6Rn10Ub/gbI1h1bGo0UrTbz05rkzw2h5sWwBZVplRW52VuZiTsTP9/PTeUy03fmrg2PT0zMDt29TzNy8h6y1c5CB6LDQ4/dvjX4QLxe'
    'XhcfwaqSU9VWmPmB5OQfTRkUSakLnrX/2r5CYGqgG7D4eMjYL/FVZiwD55uZmpuczpE0wMGbPU2luRD+O1XSykzubD9MT+fByAIDEpa0t6DlPA2VuwJEqGwQ'
    'AwClTvdV49D4O/SNbhvb41rOoNZrfkZLHJYYs9OFFmminb6ezOVm7lCJuNHs24KjW0HLJ7IxnufdwozYwnAa///XMqMBaR+YSQkXA2ejFfHMww56Etqe5DmU'
    '0XKZ6pORpx7Ee97w5kY+eTh63fDk3Zc8JJluD6rOjQ4/HP3U6gfj0Guq2AsAIuxDnkQdtcT1Yfump1uYJABDvO5ZeSEStgOqNN8HT/ktjvWGZQ6Bd22U0O9v'
    'PO+675jAh0cbEARa8NimWfRwBMEYHGO7cSzZNSIZ9HkHr6toFScwtSOR7x8HNNWfd9DDGe2gyFoiXC3ORHEetFaU+YVVQ/83LB01LMWD6LKsmJ28OzM7Z5Vj'
    'dmZq3IrJ0PSPg+L2zOx0LsP/4zGOZeP8G3eoTXRf8KuaNiDPCtpT8RuS7jI9OHCbA9fOjzB5EgI+WfFseNZWkcGIGtq849XE8emCPUIfyC5Bo83i5mJ2JfUi'
    '0b2HvjbjvwBQSwMEFAAAAAgAAAAhXIcFLGcnCQAAFBgAABMAAABNTlQwMl9NMDlfbGVzc29uLm1k7VhNbxvXFd3zVzzAmwRgROsDSatNoCZBYaAuiqCrFoWl'
    'ykor1LICSTAQLwJKGgm0KFcfJpWhPGTJRrKlhG1G5EikUKoL/ZR4N+/Nf8i5975HDv3VdtcA2ZjyfLx377nnnnve3FC3f/3bm2Pq9s2fq+/zJTU19amKu6EO'
    'faUv8sl+QcXtni4e5fBjNreVqTXjdl0Zv2C6DfwcmfAET3rGqyod7ihTLpi6Z59Ktrd1tae/6yZeSG/qdmTq5Uzmxg1l1leTyikWVfqb02QvyGTiMI+Lblf9'
    'ODTViN5zixcjXbw0tV3dCIwXKFPwzWHpuqOLJ+bvG7zeenDdsTvHnV58Furjnn09KZ/qYgF5jEiGUd7Ud81Xu7SSJHjdGcrQVPJ2i+Fc00v6jMSbEqYn+zkn'
    '5UjpHTy2oZLDE9qR7q6F5iJIygHFJag83dXtLm2cHDxSulUygUeRJDuPAE8m8557Iil7eEhhJ7PhjyiKoBUpUzwyW+eSlt4q2XUGufffB07rBfXL+RWV/LVr'
    'vDM1bopBHHov8iVao9uIw33Aj+Q8XfRf5MuDnWXNSdyqqffjMDBPsc9hZPxVFbdCmxkqiLLG5+F1Z+oXHyn95CzuPqJNq8jZ1kq3PQctgkgOjrO2ErYCWVqP'
    'qCCLmaCHyvexjygeU+2aKlYJ/41spbj93Xj5dG3iNkH2zsN3GVAXID0m2yupFGfupdOjbbmEXG+9f+RuOP608uZZ1ZZRmaiqLz1zeJqll/Cn0tt5dJQAnHUN'
    'lBxuU2jBiRrHfUsKZZ7n0QsUIGiNiqPAvTjcBXOyfKULQM3RnvzWjvGrEJ1+Eujn5xw1dnzWdIx6T/1eePIHJn2fjNcdu51tLW58IgJaWVC1d7KDd4aobaq+'
    'zdrWPCtZ0PZ/e/SaZIazaHuEum2Hl+IPgFNyUEA9kf5IinbcK7rZm1RLc58vLq0s5xbur9wcu7O0OH9n5v7MvS+W55dHFu5KG9WDuH2uSCy+3qauwf+STbqw'
    'R2QHLfRadN35aPHezB+paajZ3EXpIaU3uslWV1+suitJySPlqVdxWf3u1m84tqsSYp9U08tLszkXhI1rZmZp5PMvpkfQquem2MTPCSpBzRGfNxPfw7r1PGro'
    'ZKT1rYskeeqpqVssQBclJkE7H7euhtq4L4+Taiiw6bszKzO5Tz+Z+vj2J0BjmuXqpIetlKleIQ27xPgHYAZiB65N8PC6M/ozdwEEKAYWu+vO2Ci4Do0+0Y9R'
    'hNqGrh9ToZADKDKsLMUA+CpzsGeKVWDCc+UO5sqdzxaXFqg20zwh7i7OLudsDYlIpGEVj+si5UmKKLzUsRrpVsGqHD1rsRQEqbBgpan23loBeoqxx1NzD6aZ'
    'DZ4pX02qz2azD9X3m/vqHTS10zLLa2Sj/9nU9QJiwOaO53ZI9Zv83RFHto9zU7nl+T/l5lfmFpZZMsq1YZ1gDQlemRGi7aCBPg37uzP6rHcgH8rhpoiPpsLK'
    'W1bviAmhr7H9YF6SyBKKwjeaArYbmKxUQjsIb48SpLN/npv9y+AxS8cHM/fmQaW5AV+Haj1UByYs89c7442YxsQSiroR2SVJAxvRMKdJ8bEvD0M7qyIaZrQi'
    'AI7Pd1ODm/qwHqA1SFGTCs3+dpfESoY9YZds1tFX/pE+9nl6Q7eofkGd+2DtJL5cdUkQv5JyhXo9qQR6p0JXbti/R1kSwZPk8DFLGXcJSrcL/cpkrCaJc4Fx'
    'gF8iiWKASZi5eJRT9Yp724koiODGh+K5Cb1sR4BDPzkyB/SHK+8OerqKS/rgX3FrG1GVkWhSrseX2x8yaTiStKu5hcWVft6Iu11rHFwkpDQBCA/Z52VAmVPd'
    'Ktt5kaprCoAxMYOiQpyXrVgm8wY9e1XHBnbA8SDg6SmrAjSAa7UE71P4pr6KnkplNcSSgecg8PTzbWsJMPORGs/w685Dxh1p8QwhOKml1pqOhGIpzCZE7kgR'
    '4KCI7MJalK5OKn5LxeBlmMaFJ8I/sNI0Qkas1hQMQBXLQsZQYkurvnswTXMkCXBgFhAR5/ZwkB4UZmGGQYdFdZ6EorZvsNJYQ+Km6IfUMsY70t9sgApSTOPV'
    'k/XqeHwWcYOsrzIbHF04JDYInJJzPqhiBX1ZPAdgYKjTJkfO1/FoggEy5wF5dCGt9JOd5v1eEhVkkTjKU2dz4UUKXNQyil+WgIig8Cy2ARvqAX++6+piE09h'
    'gnQbbH3sEmbrklrBWlFh8dBEGyLbyvj7X06MWh1/lW9gSroYKxNjX0588JI7HKzLCUttnXEfHetLApTmkM5FFHrN7ztkwNOpo0hyH1TsM0HKgmoPprt1j1Cm'
    'NN0BQ8qJmmreDjPpAtCAGWnhwbZrMJmkTTsVfYzOwSjz6tbbUan4bFAZltMIppF2lSH2dQ9Z4pyFIv+E5f+IpVCZe6XioSttZ5DnmX6rASaDw4bL2ingk02N'
    'ThxoTesfaIPqHho5y60ZFfD/tE5lnYRDXuMOTov4Rz/r0SBJjea0OxfjLzGSuRftZO9/4cE98pAEUowCnubDEaKD4ltZsCpKi2JtOqBDj23cMtlTzYoteEQG'
    '/K2ikaYEYUuNzaixKHCn21lBBxIcMPo+alBn6zhZqYRTVsL4hCiD09VgdIS/l4g/mPx//aJAE5XOf4clmrzi3Qd1oO8m7BG4CYi4euuYTXx9l2g394Cat7ZB'
    'Ol0/tp6ZpwRPkv5M6PIh3BeHJKXEof8WfbioeU4LBCke9lKv/mzKjFks2QRMqjf6eCrI3AMkmR6gEtWLfOnHb+Vf5MmRmYtT0zijFciRt74VSy3diJrZdgSL'
    'c9KSeMWaq8G0p3PAU+9Xc8vLi/epMmy4ncfxpKXsYQB+NPSNv5qqx7itB0c/+aP/gmAlpM96q0ZwNfA+InuXPlYAcwffeEQUc6KJ/eOMdYEpoH+aav/9VOPh'
    'A0dQO84NzCfCtV9qG4E7vkm0/CXXeuMBOSeEnCLu/+FLUFbGYO7V7xDUFNIFPCGxFat+HO4zAPCV0lOkYZc+bnO8cpAkKL4KgZre9R0PXuMydbNLnhQX3Bay'
    'Ic8iAZmlrIzJktQKdiwNEs2kvtO+5TNtThaUr7X9j7Uj9qwtxyZkK0c2vRYJcUTmrDDhkBTUqdEG/v0HUEsDBBQAAAgIAAAAIVyxBwko6AYAAJ8NAAAcAAAA'
    'TU5UMDJf6rO17Ya1X+yekeyEseyViOuCtC5tZG1WXU8bRxR9968YKe+YkI+mjyihLVJAUT4UqS8RTWmCRHEERFX7tIY1MrZTTPDGa7LrLI2NgS7qYhtsFPPi'
    'n+OZ/Q89d2bWXj4kPizvzp1zzz333HuLzcw+H59gg9ZpuHHKRHVbmCdMWFm+1k4kBoEh1tNMZG2+7/MzQ2x8YPxDINx2aAZ8z6EnYrck7Kzo7vU7ouoPWh7j'
    'Z6Yw3X6H5w/EPxmGCOG6wwZNI9zw8FLuXLhdNugigCtck4l8TeROQ8vmuVIsugwS2hYLSyauEp4b5rqimuH5LM/Xxpj4XOStrnBtFn7M8qPDfmf22Wy/88Oz'
    'h7gDsM5tiQApUNzBaVE4e2L3kKmc4oFwv+cAtcELRr8zaJnCukCMckY025RUnZ8DZbXIvxZ4vcfoqFcHXtEwwFQO4HWoROLWLTY5zcKKg5skNMsEFrDbE56V'
    'SNweY4vzKyupJVzTDUQN5y9KfMeRYd0LSR7lhJ82olj4FFpgec0HSEBxbe53+x1cIRxTVE3GmyV8QrSmIfZdxoND3rSIVH6AGxHeS4/gMZ4/5S0j9pi3a8Lp'
    '8UNctu2IvBNah1EqE2MsgpivhZ82Ub8AaRBS3mqLvTYIILyT06Jc1Oz0O6ENWL7OCoA9R3H5uRjVt2UMmhey7N7lG6M6kJ6axzjT/DcGGGhRiBi+O2NSfJ4V'
    'ozAsFITbIzw8sIWdDq02kPkqgdhNxN/XHuLhFnmYCdML1112Z3CCE26bBOKaQB3lRaJYC5BfpiByNRJihdQi9oIYpLuAVLX50TdeVHKChKEBoqti6rwk+cMS'
    'q3bTTOp0YkUCIXlnqIMRUY4H6SmVexZU3mrzug1E4tShhlPJS4mc27JgaweD83QE817UOyDV4l82dZ0RTxVAFRph/+vyvC//o7vlTdQ78ka86RYYPcnYTJQD'
    'nCHEqqljUF++/TP5U+qP5Mu3c6sov8qcXgt3P4wuDBweOMmhQagswkKRDAQEQtmDk2DUseRLaV12SXDkGbAaF3wKF8reyvD9C61dKSDHpFYaNNfxLj8MNHgq'
    'c9YRpoNqkstoqbThh6AZ/VtB/ygHo+Qhh62K8EwQw4sk6MGpDxHgg7DwZWXYaVLmI2aVynjTxFnqVtlKuZIOXy2SS0lONapyUTTSwttW1MZrrysn9Tpzt9+Z'
    'uY/fB3yLIsOvyCrARbpNIZAVgGXtS+qXJqUQKIdqw2cTCRUYUNez7MeFVRb+3dVUTjwYuwcdDgLzmvsycn569JF4RALdPfoMS9lzOBhc82+ySdihJxs374AV'
    'pI4+yIpPbSTB0b4oT7fAv/gYOVIiZD5hxSbbKBd5KcA7uIQeSUx4HcXAuUaWGgKRYoZ3dAhjY0rJyGXQ6ao6XxMqoqPHMSZ8xJTeVOkCI3qPUIFFyS4bfVXv'
    'Uf/nvZvnAMiFZJPU+p6VVNVKhlublCzuxh0kgDOTHx1QpKPMFSuEWMsUlyYv2kYNUrq57pE9kdayEDEGEwYN+rFgkNQkISzcPVDq0tNbNW2sdlppeV9brh7i'
    'cDTdNLttMs8ozLD/b5yZGL4stLsoEORCmipnuA+iGtuy8Qw13fWg4sGneBtDiGobwDnJ+ZkTCXHmO9n5st9IrbBxQHk2PfPi8eTzqUdaeHFEul+OQ8vTIxpl'
    'Qj2vtBnF8mhERmPceTw1+XR26umr6dknL55H2jXp8FWRUB2yGBbqEoxihBTlNglrtF7gxjNTeUw00dCTWAeqGJOw/X03quDILaCCHaA+w4RwKJrKVxXVhRd5'
    'k9OkQcUGzQPI8KbGgo1isA0tlLA0alIeEC9VCjKmSrmkCBJPtQ7kugg0vXUd9bIm+/f4ci6yrkPLN0R5I657vcvFNrafp5+A7IepxblfyJJsn3u18KOBnHim'
    'i+f8LK284/KIxsTE4A0/mxxt3TB4Y1NkDOVp798tpuZ+XUnKxXXsr4V3OM8bBb5zPDwMxJerrkYlih1tJlQ805HUhJVS5DPDHQcVkSSRJLnfi+lsYWlldfn9'
    '69XUMhNBl5xMjxgDr9IBSAJrGtL/5g/HDxE4hINaji6lE5J9SiJmKNf3SorSRAVlaaPNV5o0dY+aNIlEnPqhhcccXMobmz72pDjbUKRHywq1BqiS7+rjtMJl'
    'EUStduqeawrWmWwUyEjkaeIxD2s+J0J0hobqq3jXeg4mLeaNxO3jr9qtNLVq586VVlLvl1/Pv0otL7xZWJpbXEm+WVh9tTz/2/zy/BK+nxifuH97fPx70scl'
    '/yey1rH4HDydmnw0M4XAKouF39+llldp6O6coMV1Ulenm1retXtTgeXeRYts0b46bV3j9jjtMfXe1XnLoi+caOOw7+BNNQzQXphRVHtYwHbx6qqX+B9QSwEC'
    'FAMUAAAICAAAACFc/g9WxNEGAADXDAAAGwAAAAAAAAAAAAAApAEAAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kUEsBAhQDFAAAAAgAAAAhXEfLYvCS'
    'BwAArxEAABIAAAAAAAAAAAAAAKQBCgcAAE1OVDAyX00wOV9mb3Jtcy5tZFBLAQIUAxQAAAAIAAAAIVyHBSxnJwkAABQYAAATAAAAAAAAAAAAAACkAcwOAABN'
    'TlQwMl9NMDlfbGVzc29uLm1kUEsBAhQDFAAACAgAAAAhXLEHCSjoBgAAnw0AABwAAAAAAAAAAAAAAKQBJBgAAE1OVDAyX+qzte2GtV/snpHshLHslYjrgrQu'
    'bWRQSwUGAAAAAAQABAAUAQAARh8AAAAA'
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 상황: 어떤 문제를 해결하나요?

**미션 개요**

AAR 관점에서 지난 재고 운영의 과다·과소 재고와 수요 예측 오차를 품목군, ABC 등급, 생산 상태별로 분석하여 원인을 진단하고, 앞 6개월 자료를 통해 ABC 등급별 안전재고 계수(z)와 품목군별 편향 보정값을 재보정한 뒤 뒤 6개월에서 검증하여 도입 효과를 산출하는 실습입니다.

**배경 및 필요성**

시스템을 장기 운영한 후에는 실제 성과를 사후 검토(AAR)하고, 예측 오차의 원인(예: 특정 품목군의 예측 편향, 일괄적인 안전재고 계수 적용 등)을 파악하여 파라미터를 지속적으로 최적화해야 합니다.

재보정된 파라미터가 실제 운영 비용 절감과 정비 대기시간 단축, 재고 회전율 향상에 얼마나 기여하는지 정량적으로 증명해야 현장 도입의 타당성을 확보할 수 있습니다.

**핵심 과업**

- AAR 관점의 편차 원인 분석: 앞 6개월(t=36～41) 운영 결과를 바탕으로 품목군, ABC 등급, 생산 상태별 예측 편향, 과다 재고, 결품 품목을 분석하여 오차 발생 원인을 도출합니다.
- 안전재고 파라미터 및 편향 재보정: 앞 6개월 데이터를 활용하여 ABC 등급별 안전재고 계수(z)와 품목군별 예측 편향 보정값을 새롭게 정하고, 이를 뒤 6개월(t=42～47) 데이터에서 엄격히 검증합니다.
- 3대 정량 효과 및 민감도·로드맵 산출: 현행 정책 대비 총비용 절감액, 정비 대기시간 단축률, 재고 회전율의 3대 효과 지표를 산출하고, 단가 민감도 분석 및 단기·중기·장기 도입 로드맵을 수립합니다.

**검증 및 산출물**

**검증 기준:** 편차 원인 분석 완료, 재보정 파라미터의 뒤 6개월 검증 결과 제시, 총비용·정비대기시간·회전율 등 3대 정량 효과 지표와 민감도 및 단계별 도입 로드맵 도출 여부 확인.

**주요 산출물:**

reports/mnt02_roi_analysis.md

## 📁 이번 미션에서 완성할 것

- **미션에서 완성할 내용**
  - 앞 6개월 운영 결과를 품목군·ABC 등급·생산 상태별 예측 편향, 과다 재고, 결품 품목으로 분석해 원인을 찾는다.
  - ABC 등급별 안전재고 계수(z)와 품목군별 편향 보정값을 앞 6개월로 정하고 뒤 6개월에서 검증한다.
  - 총비용, 정비 대기시간, 재고 회전율 3대 효과 지표와 단가 민감도, 단기·중기·장기 로드맵을 정리한다.
- **구현 계약**
  - 핵심 구현 계약 src/analysis/mnt02_aar.py / ev: fc,z → (뒤6개월 검증 시뮬레이션 결과,가상 총비용). 제공 D/A/sig/items로 앞6개월에서 정한 파라미터를 평가하며 검증 자료로 다시 보정하지 않는다.
- **변경 과제와 검증**
  - 추가 변경·검증 AAR 관점에서 과다·과소 재고와 수요 예측 오차를 분석하고 안전재고 파라미터를 재보정해 도입 효과를 산출한다.
  - 앞 6개 월만으로 ABC·편향·z를 정했는지 검사하고 결품 손실 가정 변경의 뒤 6개 월 결과를 비교한다.
  - 핵심구현 후첫성공실행을보존하고추가변경·재실행·실제AI원문·근거검토3건·보완·최종판단을 작성합니다.

**제출 경로:** `reports/mnt02_roi_analysis.md`, `forms/MNT02_M09_forms.md`


## 🔍 교관 검토 기준

### 정량 평가와 측정 근거

- [정량적 평가] [필수] AAR 재보정·효과 분석 — 편차 원인 분석, 재보정 파라미터의 검증 결과, 3대 정량 효과 지표와 민감도, 단계별 도입 로드맵을 제시했는가.
- PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다.
- 목표미달·0기준·미측정은그대로표시합니다.
- 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

### 정성 평가와 검증 근거

- [정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.
- PASS 증거 앞 6개 월t36～41에서 ABC·편향·z를 결정하고뒤t42～47에서 검증합니다.
- 원본의ABC 전체 12개 월 사용과미래 예측 평균을사용한sigma를 보완했습니다.
- 단가·결품 손실·대기시간은가상 가정이고 실제 예산·가동률 개선 효과가 아닙니다.
- 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

### 진행 추적 및 완료 기준

- [진행 추적 및 완료 기준] 정성·검증: 앞 6개 월t36～41에서 ABC·편향·z를 결정하고뒤t42～47에서 검증합니다.
- 원본의ABC 전체 12개 월 사용과미래 예측 평균을사용한sigma를 보완했습니다.
- 단가·결품 손실·대기시간은가상 가정이고 실제 예산·가동률 개선 효과가 아닙니다.

### 제출 파일 및 완료 기록

- 파일: reports/mnt02_roi_analysis.md, forms/MNT02_M09_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
- 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
- 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

> **주의:** 아래 구현 코드는 원본 교관용 Workbook의 참고 구현입니다. 실행 결과와 기준 달성 여부는 실제 환경에서 독립 검증해야 합니다. 승인·발주·실측 사실을 생성한 것이 아닙니다.


## 📋 순서대로 할 활동

### 활동 1: 계획과 사전 판단

- 원문 목표 및 재보정 데이터 범위 기획 원본 목표 및 기본 제공 코드를 철저히 분석하여, 사후분석(AAR) 관점에서 과다·과소 재고와 수요 예측 오차를 분석하고 안전재고 파라미터를 재보정하는 전체 시뮬레이션 흐름을 기획합니다.
- 특히 재보정 과정에서 뒤 6개월의 정보가 사전에 유출(Data Leakage)되지 않도록 엄격하게 차단하는 검증 방안을 수립합니다.
- AI 위임 범위 및 검증 계획 설정 AAR 분석 및 파라미터 재보정 과정에서 AI에 위임할 코드 작성 범위와 활용할 자료를 구체적으로 확정합니다.
- 앞 6개월 동안 도출된 파라미터가 뒤 6개월의 검증에 올바르게 적용되는지 확인하고, 본인의 사전 판단 및 검증 계획을 철저히 준비합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M09 계획: AAR 관점에서 과다·과소 재고와 수요 예측 오차를 분석하고 안전재고 파라미터를 재보정해 도입 효과를 산출한다.에 필요한 입력·기간·단위·사용 가능 시점과 ev의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2: AI와 코드 변경

- **함수 구현과 첫 실행**
  - 핵심 함수(ev) 구현 및 첫 실행 보존 fc, z를 입력받아 뒤 6개월 검증 시뮬레이션 결과와 가상 총비용을 도출하는 핵심 함수 ev를 구현하고 저장한 뒤, 첫 성공 실행 결과를 온전히 보존합니다.
  - 제공되는 데이터(D, A, sig, items)를 바탕으로 앞 6개월에서 정한 파라미터를 평가하며, 검증 자료를 활용해 임의로 다시 보정하지 않도록 엄격히 준수합니다.
- **AI 활용과 변경 과제**
  - AI 기획 반영 및 변경 과제 수행 앞서 수립한 AAR 분석 및 파라미터 재보정 기획안을 바탕으로 구현된 코드를 검증하고 AI와 함께 고도화 작업을 진행합니다.
  - 변경 과제: 앞 6개월 데이터만으로 ABC 분석, 편향, 안전재고 계수 z가 올바르게 산정되었는지 검사하고, 결품 손실 가정 변경에 따른 뒤 6개월의 시뮬레이션 결과를 비교합니다.
  - 실제 코드와 출력 결과물은 모두 안전하게 저장합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M09 구현: src/analysis/mnt02_aar.py의 ev를저장하고계약〔fc,z → (뒤6개월 검증 시뮬레이션 결과,가상 총비용). 제공 D/A/sig/items로 앞6개월에서 정한 파라미터를 평가하며 검증 자료로 다시 보정하지 않는다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3: 근거 대조와 재실행

- **변경 코드 재실행**
  - 수정 코드 저장 및 다각도 경계 시험 수정된 AAR 분석 및 재보정 코드를 안전하게 저장하고 스크립트를 재실행합니다.
  - 앞 기간(t36 ～ t41)의 ABC 분석 결과, 편향(Bias), 안전재고 계수 z, 그리고 시그마(sigma) 값을 엄격하게 고정하고, 뒤 기간(t42 ～ t47) 검증 시뮬레이션 결과와 상호 대조하여 타당성을 검증합니다.
- **AI 설명과 원자료 대조**
  - AI 설명 대조 분석 및 주의사항 준수 AI가 제시한 설명 및 코드 선택 사항 3건 이상을 실제 실습 자료, 실행 결과 로그와 상호 대조하여 검토합니다.
  - 원본의 ABC 전체 12개월 사용 방식과 미래 예측 평균을 사용한 시그마 산정 방식을 올바르게 보완했는지 재확인합니다.
  - 단가, 결품 손실, 대기 시간 등의 지표는 시뮬레이션을 위한 가상 가정이며 실제 예산 및 가동률 직접 개선 효과가 아님을 명시하고, 동일한 조건에서 철저히 재검증합니다.

#### 확인할 사항:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M09 검증: [필수] AAR 재보정·효과 분석 — 편차 원인 분석, 재보정 파라미터의 검증 결과, 3대 정량 효과 지표와 민감도, 단계별 도입 로드맵을 제시했는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - 앞 6개 월t36～41에서 ABC·편향·z를 결정하고뒤t42～47에서 검증합니다.
  - 원본의ABC 전체 12개 월 사용과미래 예측 평균을사용한sigma를 보완했습니다.
  - 단가·결품 손실·대기시간은가상 가정이고 실제 예산·가동률 개선 효과가 아닙니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4: 최종 판단과 산출물

- **산출물 제출과 상태 기록**
  - 종합 산출물 제출 및 상태 명시 원본 평가 기준에 따른 최종 결과물인 reports/mnt02_roi_analysis.md와 forms/MNT02_M09_forms.md의 실제 작성 내용, 최종 측정값, AI 대화 보존 위치, 그리고 선택 이유를 상호 연결하여 체계적으로 제출합니다.
  - 미달 항목, 미측정 항목, 문서 미작성 내용, 그리고 실제 승인 대기 상태를 명확히 구분하여 기록합니다.
- **미션별 검증 원칙**
  - AAR 분석 및 재보정 검증 원칙 명시 앞 기간(t36 ～ t41)에서 ABC 분석, 편향(Bias), 안전재고 계수 z를 결정하고, 뒤 기간(t42 ～ t47)에서 엄격하게 검증합니다.
  - 원본의 ABC 전체 12개월 사용 방식과 미래 예측 평균을 사용한 시그마(sigma) 산정 방식을 올바르게 보완했음을 명시합니다.
  - 단가, 결품 손실, 대기 시간 등의 지표는 시뮬레이션을 위한 가상 가정이며 실제 예산 및 가동률 직접 개선 효과가 아님을 최종 검토합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M09 완료: reports/mnt02_roi_analysis.md, forms/MNT02_M09_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 이 미션의 AI 활용 예시

- 가상 자료 분석·기획·코드 변경·디버깅·검증 설계·해석
- "교육용 미션 09의 핵심 함수 구현에서 누락되기 쉬운 입력값·경계조건을 알려주세요. 원본 자료에 없는 성능 수치를 만들지 말아 주세요."

AI 답변은 그대로 정답이 아닙니다. **실제 실행 로그와 최소 3건의 주장·선택을 비교**하세요.


## 원본 교관용 코드 — 참고 구현

다음은 제공된 Solution의 해당 미션 코드입니다. 실제 수행 시 환경에 필요한 패키지를 확인하고 실행하세요.


In [ ]:
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """AAR 관점에서 지난 12개월 재고 운영의 과다·과소 재고와 수요 예측 오차를 품목군·생산 상태·ABC 등급별로 분석해 원인을 찾고, 앞 6개월 자료로 ABC 등급별 안전재고 계수(z)와 품목군별 예측 편향 보정값을 재보정한 뒤 뒤 6개월에서 검증하며, 비용 절감액·정비 대기시간 단축률·재고 회전율 3대 효과 지표와 민감도, 도입 로드맵을 산출한다."""
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.stats import norm
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 야전 정비창 수리부속 청구/출고·장비 가동률 데이터 생성기
    # 실제 부대·품목·재고·단가 정보와 무관하다. 품목 40개 x 48개월(가상 2022-01 ~ 2025-12) 월별 자료를 만든다.
    # ---------------------------------------------------------------------------
    FSC = {"2520": "power_train", "2530": "brake_steer", "2910": "fuel", "2930": "cooling",
           "2940": "filter", "5330": "gasket", "6140": "battery", "5961": "semiconductor"}
    EXERCISE = {3, 8, 10}      # 가상 대규모 훈련 월(훈련 계획으로 미리 알 수 있음)
    N_MONTHS = 48


    def season(fsc, month):
        """품목군별 계절 계수: 축전지는 겨울, 냉각계통은 여름에 수요가 늘어난다."""
        if fsc == "6140":
            return 1.6 if month in (12, 1, 2) else 0.9
        if fsc == "2930":
            return 1.5 if month in (6, 7, 8) else 0.9
        return 1 + 0.1 * np.sin(2 * np.pi * month / 12)


    def make_items(n=40, seed=2):
        """품목 마스터: NSN(가상), 품목군, 장비 유형, 기준 수요, 단가, 조달 리드타임, 공급업체 수, 생산(단종) 상태."""
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n):
            f = str(rng.choice(list(FSC)))
            status = str(rng.choice(["생산중", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
            rows.append({"item_id": f"P{i + 1:03d}", "nsn": f"{f}-00-{rng.integers(100, 1000)}-{rng.integers(1000, 10000)}",
                         "fsc": f, "group": FSC[f], "equip_type": str(rng.choice(["A", "B", "C"])),
                         "base": float(np.round(rng.lognormal(2.7, 0.8), 1)),
                         "price": int(rng.lognormal(12, 1.0)) // 100 * 100,
                         "lead_time": int(rng.integers(1, 7) if status == "생산중" else rng.integers(4, 13)),
                         "suppliers": int(rng.integers(1, 5)) if status == "생산중" else 1,
                         "status": status, "alt_available": bool(rng.random() < (0.6 if status == "생산중" else 0.3))})
        return pd.DataFrame(rows)


    def make_history(items, months=N_MONTHS, seed=3):
        """월별 청구 수요(개), 긴급 불출 건수, 장비 유형별 운용시간·가동률."""
        rng = np.random.default_rng(seed)
        t = np.arange(months)
        moy = t % 12 + 1
        ex = np.isin(moy, list(EXERCISE))
        hours = {e: (100 + 20 * rng.random(months)) * np.where(ex, 1.5, 1.0) for e in "ABC"}
        avail = {e: np.clip(0.93 - 0.12 * (hours[e] - 100) / 80 + rng.normal(0, 0.015, months), 0.6, 0.99) for e in "ABC"}
        rows = []
        for it in items.itertuples():
            trend = 1 + rng.uniform(-0.1, 0.3) * t / months
            s = np.array([season(it.fsc, m) for m in moy])
            lam = it.base * s * trend * hours[it.equip_type] / 110
            d = rng.poisson(lam)
            rows.append(pd.DataFrame({"item_id": it.item_id, "t": t, "ym": [f"{2022 + k // 12}-{k % 12 + 1:02d}" for k in t],
                                      "month": moy, "exercise": ex.astype(int), "op_hours": hours[it.equip_type].round(1),
                                      "availability": avail[it.equip_type].round(3), "demand": d,
                                      "emergency": rng.binomial(d, 0.05)}))
        return pd.concat(rows, ignore_index=True)


    def make_stock(items, hist, seed=4):
        """현재(가상 2026-01) 재고: 보유량과 미입고 발주량(조달 중인 물량). 단종 품목은 추가 발주가 어려워 미입고량이 적다."""
        rng = np.random.default_rng(seed)
        recent = hist[hist["t"] >= hist["t"].max() - 5].groupby("item_id")["demand"].mean()
        st = items[["item_id", "lead_time", "status"]].copy()
        st["on_hand"] = [int(round(recent[i] * rng.uniform(0.5, 3.0))) for i in st["item_id"]]
        cover = np.where(st["status"] == "생산중", rng.uniform(0.6, 1.3, len(st)), rng.uniform(0.0, 0.6, len(st)))
        st["on_order"] = [int(round(recent[i] * L * c)) for i, L, c in zip(st["item_id"], st["lead_time"], cover)]
        return st[["item_id", "on_hand", "on_order"]]
    # ---------------------------------------------------------------------------

    # ---- 수요 예측 공통 함수(파생 피처 + 머신러닝 모델) ----
    FEATS = ["lag1", "lag2", "lag3", "lag12", "ma3", "ma6", "std6", "season_idx", "month_sin", "month_cos",
             "exercise", "op_hours", "op_ratio", "item_level", "trend_t"]


    def make_features(hist):
        """품목별 과거 수요로 파생 피처를 만든다(모두 예측 시점 이전 정보만 사용)."""
        df = hist.sort_values(["item_id", "t"]).reset_index(drop=True)
        g = df.groupby("item_id")["demand"]
        for k in (1, 2, 3, 12):
            df[f"lag{k}"] = g.shift(k)
        df["ma3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
        df["ma6"] = g.transform(lambda s: s.shift(1).rolling(6).mean())
        df["std6"] = g.transform(lambda s: s.shift(1).rolling(6).std())
        df["item_level"] = g.transform(lambda s: s.shift(1).rolling(12).mean())
        df["season_idx"] = df["lag12"] / (g.transform(lambda s: s.shift(12).rolling(12).mean()) + 1)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df["op_ratio"] = df["op_hours"] / df.groupby("item_id")["op_hours"].transform(lambda s: s.shift(1).rolling(12).mean())
        df["trend_t"] = df["t"]
        return df.dropna().reset_index(drop=True)


    def new_model():
        """Prophet/LSTM 대신 사용하는 포아송 손실 그래디언트 부스팅(개수 자료에 적합)."""
        return HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                             min_samples_leaf=20, random_state=0)


    def mape(y, p, min_y=1):
        y, p = np.asarray(y, float), np.asarray(p, float)
        m = y >= min_y
        return float(np.mean(np.abs(y[m] - p[m]) / y[m]))

    # ---- 앙상블 수요 예측·안전재고·결품(단종) 위험 지수 공통 함수 ----
    STATUS_W = {"생산중": 0.0, "단종예고": 0.5, "단종": 1.0}     # [교육용가상자료] 단종 상태 가중치


    def fit_ens(tr):
        """포아송 그래디언트 부스팅(70%) + 랜덤포레스트(30%) 앙상블."""
        h = new_model().fit(tr[FEATS], tr["demand"])
        r = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=0, n_jobs=1).fit(tr[FEATS], tr["demand"])
        return h, r


    def pred_ens(m, X, w=0.7):
        return np.clip(w * m[0].predict(X[FEATS]) + (1 - w) * m[1].predict(X[FEATS]), 0, None)


    def next_month(hist, items, t_new=48, op_mult=1.0, ex=0):
        """다음 달(t_new) 예측용 행: 작년 같은 달 운용시간 x 배수, 훈련 여부 지정."""
        last = hist[hist["t"] == t_new - 1].copy()
        prev = hist[hist["t"] == t_new - 12].set_index("item_id")["op_hours"]
        m = (t_new % 12) + 1
        f = last.assign(t=t_new, ym=f"{2022 + t_new // 12}-{m:02d}", month=m, demand=0, exercise=ex,
                        op_hours=last["item_id"].map(prev).to_numpy() * op_mult)
        X = make_features(pd.concat([hist, f], ignore_index=True))
        return X[X["t"] == t_new].set_index("item_id").loc[items["item_id"]].reset_index()


    def stock_policy(items, mu, sigma, z=1.65, review=1):
        """안전재고 = z x 월 예측오차 표준편차 x sqrt(리드타임+검토주기), 재주문점 = 월 수요 x (리드타임+검토주기) + 안전재고."""
        L = items["lead_time"].to_numpy() + review
        ss = z * sigma * np.sqrt(L)
        return pd.DataFrame({"item_id": items["item_id"], "mu": mu, "sigma": sigma, "L": L,
                             "ss": np.ceil(ss), "rop": np.ceil(mu * L + ss)})


    def risk_index(items, pol, stock):
        """결품 위험 지수(0~100): 리드타임 동안 수요가 가용 재고를 넘을 확률(정규 근사)을 기본으로,
        단종 상태·단일 공급업체·대체품 부재에 가중을 준다(가상 가중치)."""
        st = stock.set_index("item_id").loc[items["item_id"]]
        avail = (st["on_hand"] + st["on_order"]).to_numpy()
        p = 1 - norm.cdf((avail - pol["mu"].to_numpy() * pol["L"].to_numpy()) / (pol["sigma"].to_numpy() * np.sqrt(pol["L"].to_numpy()) + 1e-6))
        sw = items["status"].map(STATUS_W).to_numpy()
        sup = (items["suppliers"].to_numpy() == 1).astype(float)
        alt = (~items["alt_available"].to_numpy()).astype(float)
        score = 100 * np.clip(0.55 * p + 0.25 * sw + 0.10 * sup + 0.10 * alt, 0, 1)
        return pd.DataFrame({"item_id": items["item_id"], "p_short": p, "status_w": sw, "single_sup": sup, "no_alt": alt,
                             "avail": avail, "risk": score.round(1)})

    # ---- 월 단위 재고 백테스트(가상 비용 지수) ----
    HOLD, ORDER_COST, SHORT_MULT = 0.02, 50_000, 0.6   # [교육용가상자료] 월 보관비율(단가 대비), 발주 1회 비용(원), 결품 1개 손실(단가 배수)


    ALT_PREMIUM, ALT_LT = 1.2, 3      # [교육용가상자료] 대체품 전환 시 단가 배수, 대체품 조달 리드타임(월)


    def simulate(items, demand, fc, policy, z=1.65, months=None, sigma=None, eol=True):
        """품목별 월 재고 모의. demand/fc: (품목 x 월) 실제·예측 수요. policy: 'current'(최근 12개월 평균 + 3개월 여유 고정) 또는 'ai'(예측 + 안전재고).
        단종 품목은 원 품목 발주 불가. 'ai' 정책에서 eol=True이면 단종 품목 중 대체품이 있는 품목은 대체품(단가 x1.2, 리드타임 3개월)으로 전환 발주한다. 반환: 품목별 보관비·발주비·결품손실·불출량·평균재고."""
        n, T = demand.shape
        L = items["lead_time"].to_numpy(); price = items["price"].to_numpy().astype(float)
        can_order = (items["status"] != "단종").to_numpy()
        if policy == "ai" and eol:
            sw = (~can_order) & items["alt_available"].to_numpy()
            can_order = can_order | sw
            L = np.where(sw, ALT_LT, L); price = np.where(sw, price * ALT_PREMIUM, price)
        avg = demand[:, :1].repeat(T, 1) if months is None else months
        on = np.round(avg[:, 0] * 2.0)
        pipe = np.zeros((n, T + int(L.max()) + 1))                 # 도착 예정 물량(품목 x 월)
        for i in range(n):
            pipe[i, :L[i]] = np.round(avg[i, 0] * (1.0 if can_order[i] else 0.3))
        res = np.zeros((n, 6)); rows = np.arange(n)
        for t in range(T):
            on += pipe[:, t]
            issue = np.minimum(on, demand[:, t]); short = demand[:, t] - issue; on -= issue
            ip = on + pipe[:, t + 1:].sum(1)
            if policy == "current":
                target = avg[:, t] * (L + 3)
            else:
                s = sigma if sigma is not None else np.sqrt(np.maximum(fc[:, t], 1))
                target = fc[:, t] * (L + 1) + z * s * np.sqrt(L + 1)
            q = np.where(can_order & (ip < target), np.ceil(target - ip), 0)
            pipe[rows, t + L] += q
            res += np.c_[HOLD * price * on, ORDER_COST * (q > 0), SHORT_MULT * price * short, issue, on, short]
        out = pd.DataFrame(res, columns=["hold", "order", "short", "issued", "inv", "short_qty"], index=items["item_id"])
        out["inv"] /= T
        return out

    WAIT_H = 36            # [교육용가상자료] 결품 1개당 장비 정비 대기시간(시간)

    items = make_items()
    hist = make_history(items)
    df = make_features(hist)
    m = fit_ens(df[df["t"] < 36])
    bt = df[df["t"] >= 36].copy(); bt["fc"] = pred_ens(m, bt)
    P = lambda c: bt.pivot(index="item_id", columns="t", values=c).loc[items["item_id"]].to_numpy()
    D, F, A = P("demand"), P("fc"), P("item_level")
    sig = np.maximum(0.25 * F.mean(1), np.sqrt(np.maximum(F.mean(1), 1)))
    val = items["price"].to_numpy() * D.sum(1)
    cum = np.cumsum(np.sort(val)[::-1]) / val.sum()
    abc = pd.Series(np.select([val >= np.sort(val)[::-1][np.searchsorted(cum, 0.7)], val >= np.sort(val)[::-1][np.searchsorted(cum, 0.9)]], ["A", "B"], "C"), index=items["item_id"])

    # 1) AAR 편차 분석: 예측 편향과 과다·과소 재고
    an = items[["item_id", "group", "status"]].assign(abc=abc.values, bias=(F[:, :6].sum(1) - D[:, :6].sum(1)) / np.maximum(D[:, :6].sum(1), 1))
    r0 = simulate(items, D[:, :6], F[:, :6], "ai", 1.65, A[:, :6], sig)
    an["over"] = (r0["inv"].to_numpy() > 3 * np.maximum(D[:, :6].mean(1), 1))       # 평균 재고가 3개월 수요 초과 = 과다
    an["under"] = r0["short_qty"].to_numpy() > 0
    print("[AAR 편차 분석: 앞 6개월(t=36~41)]")
    g = an.groupby("group").agg(예측편향=("bias", "mean"), 과다품목=("over", "sum"), 결품품목=("under", "sum"))
    gt = g.T.astype(object)
    gt.loc["예측편향"] = [f"{v:+.0%}" for v in g["예측편향"]]
    gt.loc[["과다품목", "결품품목"]] = gt.loc[["과다품목", "결품품목"]].map(lambda v: f"{int(v)}")
    print(gt.to_string())
    print("ABC별 과다/결품 품목:", ", ".join(f"{k} {int(v['over'])}/{int(v['under'])}" for k, v in an.groupby("abc")[["over", "under"]].sum().iterrows()),
          "| 생산 상태별 결품 품목:", ", ".join(f"{k} {int(v)}" for k, v in an.groupby("status")["under"].sum().items()))
    ex = bt["exercise"] == 1
    hi = g["예측편향"].abs().sort_values(ascending=False).index[:2]
    hi_txt = ", ".join(f"{h}({g.loc[h, '예측편향']:+.0%})" for h in hi)
    print(f"훈련 월 예측 편향 {(bt.loc[ex, 'fc'].sum() / bt.loc[ex, 'demand'].sum() - 1):+.1%}, 평시 월 {(bt.loc[~ex, 'fc'].sum() / bt.loc[~ex, 'demand'].sum() - 1):+.1%}"
          f" -> 원인: {hi_txt} 품목군 예측 편향, 모든 등급에 같은 z=1.65 적용, 단종 품목 발주 불가")

    # 2) 재보정(앞 6개월로 결정): ABC별 z 격자 탐색 + 품목군별 편향 보정
    grp_bias = an.groupby("group")["bias"].median().clip(-0.3, 0.3)
    corr = 1 / (1 + items["group"].map(grp_bias).to_numpy())
    best = {}
    for cls in "ABC":
        mask = (abc == cls).to_numpy()
        costs = {}
        for z in (0.84, 1.28, 1.65, 2.05, 2.33):
            zz = np.where(mask, z, 1.65)
            r = simulate(items, D[:, :6], F[:, :6] * corr[:, None], "ai", zz, A[:, :6], sig)
            costs[z] = r[["hold", "order", "short"]].to_numpy()[mask].sum()
        best[cls] = min(costs, key=costs.get)
    z_new = abc.map(best).to_numpy()
    print(f"\n[재보정 결과] ABC별 z: A {best['A']}, B {best['B']}, C {best['C']} (기존 1.65 일괄), 품목군 편향 보정 범위 {corr.min():.2f}~{corr.max():.2f}배")

    # 3) 검증(뒤 6개월, t=42~47) 및 3대 효과 지표
    def ev(fc, z):
        r = simulate(items, D[:, 6:], fc, "ai", z, A[:, 6:], sig)
        return r, r[["hold", "order", "short"]].sum().sum()


    r_cur = simulate(items, D[:, 6:], F[:, 6:], "current", months=A[:, 6:])
    r_old, c_old = ev(F[:, 6:], 1.65)
    r_new, c_new = ev(F[:, 6:] * corr[:, None], z_new)
    c_cur = r_cur[["hold", "order", "short"]].sum().sum()
    turn = lambda r: float(r["issued"].sum() / r["inv"].sum())
    over_amt = lambda r: float((np.maximum(r["inv"].to_numpy() - 3 * np.maximum(D[:, 6:].mean(1), 1), 0) * items["price"].to_numpy()).sum())
    k = pd.DataFrame([["6개월 총비용(백만 원)", c_cur / 1e6, c_old / 1e6, c_new / 1e6],
                      ["정비 대기시간(시간)", r_cur["short_qty"].sum() * WAIT_H, r_old["short_qty"].sum() * WAIT_H, r_new["short_qty"].sum() * WAIT_H],
                      ["재고 회전율(회/6개월)", turn(r_cur), turn(r_old), turn(r_new)]],
                     columns=["지표", "현행 정책", "예측 정책(기존 파라미터)", "예측 정책(재보정)"]).set_index("지표")
    k["현행 대비 개선율"] = 1 - k["예측 정책(재보정)"] / k["현행 정책"]
    k.loc["재고 회전율(회/6개월)", "현행 대비 개선율"] *= -1          # 회전율은 클수록 좋음
    print("\n[3대 정량 효과 지표: 뒤 6개월 검증, 가상 단가 기준]")
    print(k.to_string(float_format=lambda x: f"{x:,.1f}", formatters={"현행 대비 개선율": "{:.1%}".format}))
    print(f"재보정 효과(기존 파라미터 대비): 총비용 {1 - c_new / c_old:+.1%}, 연간 환산 절감액(현행 대비) {(c_cur - c_new) * 2 / 1e6:,.0f}백만 원(가상)")
    sens = []
    for hm in (0.5, 1.0, 1.5):
        for sm in (0.3, 0.6, 1.0):
            HOLD_, SM_ = HOLD * hm, sm
            cc = lambda r, p: (HOLD_ * p * r["inv"].to_numpy() * 6).sum() + r["order"].sum() + (SM_ * p * r["short_qty"].to_numpy()).sum()
            pr = items["price"].to_numpy()
            sens.append(f"보관x{hm}/결품x{sm}: {1 - cc(r_new, pr) / cc(r_cur, pr):.0%}")
    print("[민감도: 현행 대비 비용 절감률]", ", ".join(sens))
    print("도입 로드맵: 단기(1~3개월) 분기별 AAR로 z·편향 재보정 / 중기(4~9개월) 단종예고 품목 최종 구매·대체품 검증 절차 정착 / 장기(10개월~) 군수 정보체계 연계와 자동 재학습")
    print("금액·시간은 가정 단가로 계산한 상대 비교값이며 예산 근거로 쓰지 않는다.")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    g["예측편향"].plot.bar(ax=ax[0], rot=30, color="tab:purple"); ax[0].axhline(0, c="k", lw=0.8); ax[0].set_title("Forecast bias by group (AAR, first 6 months)")
    k.iloc[:, :3].div(k["현행 정책"], axis=0).T.plot.bar(ax=ax[1], rot=0); ax[1].set_xticklabels(["current", "forecast(old)", "forecast(recal)"])
    ax[1].set_title("Effect indicators (current = 1.0)"); ax[1].legend(["cost", "wait hours", "turnover"])
    plt.tight_layout(); plt.show()


## 원본 교관용 예상 출력 (참고용)

아래 내용은 원본에 포함된 예시이며 **실제 실행 결과로 제출할 수 없습니다.**

<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 편차 분석: 앞 6개월(t=36～41)]
group battery brake_steer cooling filter fuel gasket power_train semiconductor
예측편향     +11%        +18%     +4%    +1%  -6%    -5%         +1%           -3%
과다품목        1           1       1      2    0      0           0             0
결품품목        1           0       0      0    1      0           0             0
ABC별 과다/결품 품목: A 1/0, B 2/0, C 2/2 | 생산 상태별 결품 품목: 단종 1, 단종예고 0, 생산중 1
훈련 월 예측 편향 +1.5%, 평시 월 +0.1% -> 원인: brake_steer(+18%), battery(+11%) 품목군 예측 편향, 모든 등급에 같은 z=1.65 적용, 단종 품목 발주 불가

[재보정 결과] ABC별 z: A 0.84, B 0.84, C 1.28 (기존 1.65 일괄), 품목군 편향 보정 범위 0.89～1.07배

[3대 정량 효과 지표: 뒤 6개월 검증, 가상 단가 기준]
                 현행 정책  예측 정책(기존 파라미터)  예측 정책(재보정) 현행 대비 개선율
지표                                                          
6개월 총비용(백만 원)    117.7            56.7        55.3     53.0%
정비 대기시간(시간)   17,532.0         1,620.0     1,872.0     89.3%
재고 회전율(회/6개월)      3.1             3.0         3.1      1.2%
재보정 효과(기존 파라미터 대비): 총비용 +2.5%, 연간 환산 절감액(현행 대비) 125백만 원(가상)
[민감도: 현행 대비 비용 절감률] 보관x0.5/결품x0.3: 55%, 보관x0.5/결품x0.6: 69%, 보관x0.5/결품x1.0: 77%, 보관x1.0/결품x0.3: 38%, 보관x1.0/결품x0.6: 55%, 보관x1.0/결품x1.0: 67%, 보관x1.5/결품x0.3: 28%, 보관x1.5/결품x0.6: 45%, 보관x1.5/결품x1.0: 58%
도입 로드맵: 단기(1～3개월) 분기별 AAR로 z·편향 재보정 / 중기(4～9개월) 단종예고 품목 최종 구매·대체품 검증 절차 정착 / 장기(10개월～) 군수 정보체계 연계와 자동 재학습
금액·시간은 가정 단가로 계산한 상대 비교값이며 예산 근거로 쓰지 않는다.
```
</details>


## 교관 검토 기록

학생의 실제 실행 로그·대화 원문·수정 이력·평가 기준 충족 여부를 확인하고, 달성/미달/추가확인 및 승인 여부를 별도 기록합니다.

**통합 입력·저장 기능:** 마지막 통합 화면에서 미션별 산출물을 파일별로 작성·수정·저장하고, 실제 AI 질문·답변 원문, 원자료·출력 대조 근거, 검토 3건, 수정과 최종 수행 상태를 기록한 뒤 ZIP으로 다운로드합니다.

저장만으로 PASS나 승인이 되지 않습니다.


In [ ]:
# @title ⑤ 교관용 개별 산출물·검토기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 9
        DELIVERABLE_PATHS = ['forms/MNT02_M09_forms.md', 'reports/mnt02_roi_analysis.md', 'src/analysis/mnt02_aar.py']
        PROJECT_NAME = 'MNT02'
        NOTEBOOK_ROLE = '교관용 Solution'
        WORK_ROOT = Path(MNT02_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'MNT02_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'MNT02_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','실습 계획·검토 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# MNT02 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('실습 계획·검토 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(v['dirty'] for v in editors.values()):
                    print('미저장 변경이 있습니다. 먼저 저장 후 ZIP을 다시 만드세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ MNT02 미션 {MISSION_CODE:02d} ({NOTEBOOK_ROLE}) 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 교관용 해설 및 예상 출력은 실제 실행·승인 증거가 아닙니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
